# Learn Spark

## Setting Up Spark

### Install PySpark

In [0]:
!pip install pyspark

### Setting Up PySpark Session

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("Colab_PySpark_Test").getOrCreate()

### Importing Libraries

In [0]:
from pyspark.sql.functions import *
import pandas as pd

In [0]:
from google.colab import drive
drive.mount('/content/drive')

## Sample Codes

#### Merge two Dataframes using PySpark


Here we use union() to merge two DataFrames. In there is a schema mismatch, then the table that is missing col.s are filled with null value using the function lit().

In [0]:
simpleData1 = [(1,"Sagar","CSE","UP",80),\
    (2,'Shivam','IT','MP',86),\
        (3,'Priyanka','CSE','UP',85)]

columns = ["Id","Name","Dept","State","Score"]
df_1 = spark.createDataFrame(data = simpleData1, schema = columns)
df_1.show(5)

In [0]:
simpleData2 = [(4,'Priyanshu','IT','UP'),\
                (5,'Neha','CSE','UP')]

columns = ["Id","Name","Dept","State"]
df_2 = spark.createDataFrame(data = simpleData2, schema = columns)
df_2.show(5)

In [0]:
df_2 = df_2.withColumn("Score",lit(None))

In [0]:
df_1.union(df_2).show()

#### Explode columns using PySpark

explode() function is being used here which is used to transform an array i.e. list or map i.e. dict column into multiple rows. Each element in the array or map becomes a separate row in the resulting DataFrame.

Important Links:
<br>
- [https://rajanand.org/spark/spark-explode](url)
- [https://towardsdatascience.com/pyspark-explained-the-explode-and-collect-list-functions-834f45ff5ac5/](url)
- [https://bigdataenthusiast.medium.com/apache-spark-explode-function-f8c0ef87452](url)
- [https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.explode.html](url)

In [0]:
# Exploding a list
data = [("Anand", ["Java", "Python"]),
        ("Bala", ["Scala", "Spark","Azure"]),
        ("Kavitha", ["SQL", "Hadoop"])]
columns = ["Name","Skills"]

df = spark.createDataFrame(data, columns)
df.show()

exploded_df = df.select("Name", explode("Skills").alias("Skills"))
exploded_df.show()

In [0]:
# Exploding a Dictionary
data = [("James",{"Java":2,"Scala":2,"Python":4}),
        ("Michael",{"Spark":5,"Java":4}),
        ("Robert",{"CSharp":5}),
        ("Washington",{"C++":5})]

columns = ["name" ,"SkillLevel"]
df = spark.createDataFrame(data, columns)

exploded_df = df.select("name", explode("SkillLevel").alias("Skill","Level"))
exploded_df.show()

#### Regex using PySpark

Here we are using rlike function which is similar to LIKE in SQL. It is used to filter DataFrame by matching a col.'s values against regular expression.

In [0]:
sampleData1 = [
    (2, "Shivam", "U6754555iy"),
    (1, "Sagar", "2872587258"),
    (3, "Muni", "86468286U7")
]

columns = ["Id", "Name", "Mobile"]

df1 = spark.createDataFrame(sampleData1, columns)
display(df1)

In [0]:
df1.select("*").filter(col("Mobile").rlike("^[0-9]+$")).show()

#### Skip line while loading data into DataFrame

Here we will see methods to skip rows while loading data into PySpark DataFrame.
Mainly there are 3 methods:
1. Using skipRows
2. Using RDD
3. Using Pandas

In [0]:
# Using skipRows
df1 = spark.read.option("header","true").option("skipRows",4).csv("/content/drive/MyDrive/Colab Notebooks/Spark/Data/SkipLineTestData.csv")

In [0]:
df1.show()

In [0]:
# Using Pandas
dfpd = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Spark/Data/SkipLineTestData.csv")
df1 = spark.createDataFrame(dfpd[4:], schema =list(dfpd.columns))
df1.show()

In [0]:
# Using RDD
rdd = spark.sparkContext.textFile("/content/drive/MyDrive/Colab Notebooks/Spark/Data/SkipLineTestData.csv")
rdd_skipped = rdd.zipWithIndex()
rdd_skipped.collect()

In [0]:
rdd_skipped = rdd_skipped.filter(lambda x:x[1]>4)
df1 = spark.read.option("header","true").csv(rdd_skipped)
df1.show()

#### Count Nulls in each column

Here, we are calculating the null count in each column of a DataFrame.
Here we are using isNull to check for Null values and to count Null values we are using when which returns 1 only when condition is true.

In [0]:
df1 = spark.read.option("header",True).csv("/content/drive/MyDrive/Colab Notebooks/Spark/Data/RowCountTestData.csv")
df1.show()

In [0]:
df1.select([count(when(col(i).isNull(),1)) for i in df1.columns]).show()

#### Handle multi delimiters

Here split() function is being used to handle multi delimited data. It takes 2 arguments, data and separator, and returns delimited data in list format.

In [0]:
df1 = spark.read.option("header",True).csv("/content/drive/MyDrive/Colab Notebooks/Spark/Data/Multi-Delimiter.csv")
df1.show()

In [0]:
df1 = df1.withColumn('Physics', split(col('Marks'),'\\|')[0])\
.withColumn('Maths',split(col('Marks'),'\\|')[1])\
.withColumn('Chemistry',split(col('Marks'),'\\|')[2])\
.drop('Marks')
df1.show()

#### Regex Replace

In [0]:
df = spark.read.text('/content/drive/MyDrive/Colab Notebooks/Spark/Data/Regex_Replace.txt')
df.show()

In [0]:
pattern = rf"(.{{{7-1}}})."  # Generates: r"(.{2})."
replacement = r"$1,"          # Keeps first 2 chars, replaces 3rd with *
df1 = df.withColumn('values',regexp_replace(col('value'),pattern,replacement)).drop('value')
df1 = df1.withColumn('values',explode(split(col("values"),',')))
df1 = df1.withColumn('ID',split(col('values'),'-')[0]).withColumn('Name',split(col('values'),'-')[1]).withColumn('Age',split(col('values'),'-')[2]).drop('values')
df1.show()

In [0]:
df1 = df.withColumn('values',regexp_replace(col('value'),'(.*?\\-){3}','$0,')).drop('value')
df1 = df1.withColumn('values',explode(split(col("values"),'-,')))
df1 = df1.withColumn('ID',split(col('values'),'-')[0]).withColumn('Name',split(col('values'),'-')[1]).withColumn('Age',split(col('values'),'-')[2]).drop('values')
df1.show()

#### Pivot and Explode

Here Pivot and Explode is being used to Pivot a particular dataset on Pivot column.
<br>
Pivot = Row -> Column
<br>
Unpivot = Column -> Row

To pivot a DataFrame on particular col.:
<br>
df.groupBy(<Unpivot Cols>).pivot(<Pivot Cols>).agg(collect_list(<Value Cols>))
<br>
Here, arrays_zip() is used to collect all the pivot cols so that explode can be applied on a single col.


In [0]:
df1 = spark.read.format('csv').option('header',True).load('/content/drive/MyDrive/Colab Notebooks/Spark/Data/Pivot_Dataset.csv')
df1.show()

In [0]:
df2 = df1.groupBy('ID', 'Date_part').pivot('Name').agg(collect_list('COUNTRY')).orderBy('ID')
df2 = df2.withColumn('new', arrays_zip("Bey","Gaga","Katy"))
df2 = df2.withColumn('new1', explode("new")).drop("new")
df2 = df2.select("ID","Date_part",'new1.Bey','new1.Gaga','new1.Katy')

In [0]:
df2.show()

In [0]:
df3 = df2.unpivot(
    ids=['ID','Date_part'],
    values=["Bey", "Gaga", "Katy"],
    variableColumnName="Name",
    valueColumnName="Country"
)

In [0]:
df3.show()#filter(col('Country').isNotNull()).show()

#### CollectSet

Here CollectSet is being used to group unique list of addresses into list for each custId.

In [0]:
dataset = [{'custId':1, 'custName': 'Mark Ray', 'address':'AB'},
           {'custId':2, 'custName': 'Peter Smith', 'address':'CD'},
           {'custId':1, 'custName': 'Mark Ray', 'address':'EF'},
           {'custId':2, 'custName': 'Peter Smith', 'address':'GH'},
           {'custId':2, 'custName': 'Peter Smith', 'address':'CD'},
           {'custId':3, 'custName': 'Kate', 'address':'ID'}]


from pyspark.sql.types import *
schema = StructType([
    StructField('custId', IntegerType(), True),
    StructField('custName', StringType()),
    StructField('address', StringType())
])

df1 = spark.createDataFrame(data=dataset, schema=schema)
df1.show()

In [0]:
df1.groupBy('custID', 'custName').agg(collect_set('address').alias('address')).orderBy('custID').show()

#### Regexp Extract

Here regexp_extract function is being used to extract specific substrings from a column.  

In [0]:
data=[('ABSHFJFJ12QWERT12',1),('QWERT5674OTUT1',2),('DGDGNJDJ1234UYI',3)]
df=spark.createDataFrame(data,schema="input_string string,id int")
df.show()

In [0]:
df.select("*").\
withColumn("new_col1",regexp_extract(col("input_string"),'(^[a-zA-Z]*[0-9]*)',1)).\
withColumn("new_col2",regexp_extract(col("input_string"),'([a-zA-Z]*[0-9]*$)',1)).\
show()

#### Next Question